In [19]:
import os
import pandas as pd
from hashlib import md5
import torch

In [41]:
data_path = "/srv/defectDetectionDataset/multiclassClassification/full_clean_clustered_new"

In [42]:
images = []

extensions = [".png", ".jpg", ".jpeg"]

for dirpath, dirnames, filenames in os.walk(data_path):
    split = "train" if "train" in dirpath else "val" if "val" in dirpath else "test"
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            label = os.path.basename(dirpath)
            image_id = md5(filename.encode()).hexdigest()
            images.append({"image_id": image_id, "label": label, "split": split})

df = pd.DataFrame(images)

In [43]:
df

,image_id,label,split
0,ab668d6568f9b35f9dc6585662ed35d5,missing_part,test
1,753ca10ded3097485798a381d2ce2a41,missing_part,test
2,20f6d9d52c90b66b68eed4cb0a797f22,missing_part,test
3,e849c3a5d52bc31b8c34fd5dcec15028,missing_part,test
4,34634b27c2b4aecda5facfbc583167bf,missing_part,test
...,...,...,...
5280,47eea0d7c1ca044008f99623c995dadb,other,train
5281,0d963170b96e3c4227e2c681128602a7,other,train
5282,51b685eefe40f5b1aa2b24466d8516a8,other,train
5283,68c7f62c8cab3a91a840a7a7e222c15d,other,train


In [44]:
noisy_path = "/srv/defectDetectionDataset/multiclassClassification/noisy_new"

In [45]:
import shutil

def list_image_files(root_dir, exts=(".png", ".jpg", ".jpeg")):
    files = []
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.lower().endswith(exts):
                files.append(os.path.join(dirpath, filename))
    return files

train_dir = os.path.join(noisy_path, "train")
val_dir = os.path.join(noisy_path, "val")
test_dir = os.path.join(noisy_path, "test")

train_files = list_image_files(train_dir)
before_train_count = len(train_files)
before_total_count = sum(len(list_image_files(d)) for d in [train_dir, val_dir, test_dir] if os.path.isdir(d))

df_lookup = df.set_index("image_id")[["split", "label"]].to_dict("index")

moved_to_val = 0
moved_to_test = 0
removed = 0
kept_train = 0

for file_path in train_files:
    filename = os.path.basename(file_path)
    image_id = md5(filename.encode()).hexdigest()
    record = df_lookup.get(image_id)
    if record is None:
        os.remove(file_path)
        removed += 1
        continue

    split = record["split"]
    label = record["label"]
    if split == "train":
        kept_train += 1
        continue

    target_root = val_dir if split == "val" else test_dir if split == "test" else None
    if target_root is None:
        os.remove(file_path)
        removed += 1
        continue

    target_dir = os.path.join(target_root, label)
    os.makedirs(target_dir, exist_ok=True)
    target_path = os.path.join(target_dir, filename)
    shutil.move(file_path, target_path)
    if split == "val":
        moved_to_val += 1
    else:
        moved_to_test += 1

after_train_count = len(list_image_files(train_dir))
after_total_count = sum(len(list_image_files(d)) for d in [train_dir, val_dir, test_dir] if os.path.isdir(d))

print(f"Before: train={before_train_count}, total={before_total_count}")
print(f"After:  train={after_train_count}, total={after_total_count}")
print(f"Kept train: {kept_train}")
print(f"Moved to val: {moved_to_val}")
print(f"Moved to test: {moved_to_test}")
print(f"Removed: {removed}")

Before: train=5315, total=5315
After:  train=3506, total=5284
Kept train: 3506
Moved to val: 815
Moved to test: 963
Removed: 31
